This notebook includes basic join operations and window functions

In [0]:
data1 = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David"),
    (5, "Eve")
]
data2 = [
    (3, "Math"),
    (4, "Science"),
    (5, "English"),
    (6, "History"),
    (7, "Art")
]

df1 = spark.createDataFrame(data1, ["stu_id", "name"])
df2 = spark.createDataFrame(data2, ["id", "subject"])

display(df1)
display(df2)

Inner Join

In [0]:
inner_df = (
    df1
    .join(
        df2,
        on=["stu_id"],
        how="inner"
    )
)

inner_df.display()

In [0]:
inner_df = (
    df1
    .join(
        df2,
        on=df1.stu_id == df2.id,
        how="inner"
    )
    .drop(df2.id)
)

inner_df.display()

Left Join

In [0]:
left_df = (
    df1
    .join(
        df2,
        on=df1.stu_id == df2.id,
        how="left"
    )
    .drop(df2.id)
)

left_df.display()

In [0]:
right_df = (
    df1
    .join(
        df2,
        on=df1.stu_id == df2.id,
        how="right"
    )
    # .drop(df.id)
)

right_df.display()

In [0]:
full_df = (
    df1
    .join(
        df2,
        on=df1.stu_id == df2.id,
        how="full"
    )
    # .drop(df.id)
)

full_df.display()

In [0]:
cross_df = (
    df1
    .crossJoin(df2)
    # .drop(df.id)
)

cross_df.display()

Window Functions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
cust_df = spark.table("samples.tpch.customer")
display(cust_df)

In [0]:
# primary key constrint  (unique and also pk can't have nulls)
# unique key constrint (distinct/unique elements it can null values)

In [0]:
check_df = (
    cust_df
    .groupBy("c_custkey")
    .agg(
        F.count("*").alias("cnt")
    )
    .filter(F.col("cnt") > 1)
)

display(check_df)

In [0]:
orders_df = spark.table("samples.tpch.orders")
display(orders_df)

In [0]:
check_df = (
    orders_df
    .groupBy("o_custkey")
    .agg(
        F.count("*").alias("cnt")
    )
    .filter(F.col("cnt") > 1)
    .sort(F.col("cnt").desc())
)

display(check_df)

In [0]:
check_df = (
    orders_df
    .groupBy("o_orderkey")
    .agg(
        F.count("*").alias("cnt")
    )
    .filter(F.col("cnt") > 1)
)

display(check_df)

In [0]:
joined_df = (
    cust_df.select("c_custkey", "c_name")
    .join(
        orders_df.select("o_orderkey", "o_custkey", "o_totalprice"),
        on=cust_df.c_custkey == orders_df.o_custkey,
        how="inner"
    )
    .drop("o_custkey")
)

display(joined_df)

In [0]:
result_df = (
    joined_df
    .groupBy("c_custkey", "c_name")
    .agg(
        F.collect_list("o_orderkey").alias("orders_list"),
        F.sum("o_totalprice").alias("TotalSpent")
    )
)
result_df.display()

In [0]:
windowSpec = Window.orderBy(F.col("TotalSpent").desc())
ranked_df = (
    result_df
    .withColumn("rank", F.row_number().over(windowSpec))
    .filter(F.col("rank") <= 2)
)
ranked_df.display()